# Two-Model Displacement Comparison

This notebook runs the same tests against two Microsoft Foundry model deployments and produces one visual displacement scorecard. It compares response quality, latency, token use, estimated cost, and run-to-run stability, then recommends **Move**, **Pilot**, or **Hold** for each workload.

The deployments are read from `MODEL_DEPLOYMENT_NAME_0` (baseline) and `MODEL_DEPLOYMENT_NAME` (candidate) in `.env`. Live inference is disabled by default.

## 1. Load the two model deployments

Set `FDKIT_ENV_FILE` to an external `.env` path or place `.env` in the kit root. Set `RUN_LIVE_EVALUATION = True` only when you are ready to send the test prompts to both deployments.

In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import html
import json
import os
import time

import pandas as pd
from dotenv import load_dotenv
from IPython.display import HTML, clear_output, display


def find_kit_root() -> Path:
    for parent in (Path.cwd(), *Path.cwd().parents):
        if (parent / "datasets").is_dir() and (parent / "notebooks").is_dir():
            return parent
        nested = parent / "kits" / "flash-displacement-kit"
        if nested.is_dir():
            return nested
    raise FileNotFoundError("Run this notebook from the repository workspace.")


KIT_ROOT = find_kit_root()
ENV_FILE = Path(os.getenv("FDKIT_ENV_FILE", str(KIT_ROOT / ".env"))).expanduser()
load_dotenv(dotenv_path=ENV_FILE)
print(f"Loaded environment variables from {ENV_FILE}")

PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT", "")
BASELINE_MODEL_NAME = os.getenv("BASELINE_MODEL_NAME", "")
CANDIDATE_MODEL_NAME = os.getenv("CANDIDATE_MODEL_NAME", "")
RUN_LIVE_EVALUATION = True
REPEATS = 3
REQUEST_TIMEOUT_SECONDS = 60.0
MAX_COMPLETION_TOKENS = 256
# Live requests in flight at once; 1 is sequential and gives the cleanest latency numbers.
MAX_CONCURRENT_REQUESTS = 8
PROGRESS_EVERY = 25

MODELS = [
    {
        "role": "baseline",
        "deployment": BASELINE_MODEL_NAME or "Baseline model",
        "color": "#b39ddb",
        "input_price": float(os.getenv("BASELINE_INPUT_PRICE_PER_MILLION_USD", "0")),
        "output_price": float(os.getenv("BASELINE_OUTPUT_PRICE_PER_MILLION_USD", "0")),
    },
    {
        "role": "candidate",
        "deployment": CANDIDATE_MODEL_NAME or "Candidate model",
        "color": "#0f6cbd",
        "input_price": float(os.getenv("CANDIDATE_INPUT_PRICE_PER_MILLION_USD", "0")),
        "output_price": float(os.getenv("CANDIDATE_OUTPUT_PRICE_PER_MILLION_USD", "0")),
    },
]

credential = project_client = openai_client = None
if RUN_LIVE_EVALUATION:
    missing = [
        name
        for name, value in {
            "PROJECT_ENDPOINT": PROJECT_ENDPOINT,
            "MODEL_DEPLOYMENT_NAME_0": BASELINE_MODEL_NAME,
            "MODEL_DEPLOYMENT_NAME": CANDIDATE_MODEL_NAME,
        }.items()
        if not value
    ]
    if missing:
        raise ValueError(f"Missing required .env values: {', '.join(missing)}")
    if BASELINE_MODEL_NAME == CANDIDATE_MODEL_NAME:
        raise ValueError("The baseline and candidate deployment names must be different.")

    from azure.ai.projects import AIProjectClient
    from azure.identity import DefaultAzureCredential

    credential = DefaultAzureCredential()
    project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
    openai_client = project_client.get_openai_client().with_options(
        timeout=REQUEST_TIMEOUT_SECONDS,
        max_retries=0,
    )

mode = "LIVE FOUNDRY CALLS" if RUN_LIVE_EVALUATION else "ILLUSTRATIVE DATA - NO MODEL CALLS"
display(HTML(f"""
<div style="border-left:5px solid #087f5b;padding:12px 16px;background:#f2f7f4;font-family:Segoe UI,sans-serif">
  <strong>{html.escape(mode)}</strong><br>
  Baseline: <code>{html.escape(MODELS[0]['deployment'])}</code><br>
  Candidate: <code>{html.escape(MODELS[1]['deployment'])}</code><br>
  Live-call guard: {REQUEST_TIMEOUT_SECONDS:.0f}s timeout, no hidden retries, {MAX_COMPLETION_TOKENS} completion tokens maximum
</div>
"""))

Loaded environment variables from /mnt/c/Users/loreaa/VS_CODE/FOUNDRY-DEEPSEEK/kits/flash-displacement-kit/.env


## 2. Load the shared synthetic test set

The notebook loads the JSONL files listed in `SYNTHETIC_DATASETS` (summarisation, classification, extraction) and keeps at most `MAX_EXAMPLES_PER_CATEGORY` rows from each, so a large corpus can be sampled down for a quick run. Set it to `None` to score everything. Both deployments receive exactly the same prompts. Live responses are scored by the expected concepts in each row; `reference_answer` and optional `illustrative_candidate_response` fields drive offline illustrative mode only. Review or replace these synthetic cases with representative workload evidence before making a production decision.

In [2]:
SYNTHETIC_DATASETS = {
    "Summarisation": KIT_ROOT / "datasets" / "synthetic-large" / "summarisation.jsonl",
    "Classification": KIT_ROOT / "datasets" / "synthetic-large" / "classification.jsonl",
    "Extraction": KIT_ROOT / "datasets" / "synthetic" / "extraction.jsonl",
}
REQUIRED_TEST_FIELDS = {"id", "category", "prompt", "expected_keywords", "reference_answer"}


def load_test_cases(dataset_files: dict[str, Path]) -> list[dict]:
    test_cases = []
    seen_ids = set()

    for expected_category, dataset_path in dataset_files.items():
        if not dataset_path.is_file():
            raise FileNotFoundError(f"Synthetic dataset not found: {dataset_path}")

        with dataset_path.open(encoding="utf-8") as dataset_file:
            for line_number, raw_line in enumerate(dataset_file, start=1):
                if not raw_line.strip():
                    continue
                try:
                    test_case = json.loads(raw_line)
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSON in {dataset_path.name} at line {line_number}") from exc

                if not isinstance(test_case, dict):
                    raise ValueError(f"Expected an object in {dataset_path.name} at line {line_number}")
                missing_fields = REQUIRED_TEST_FIELDS - test_case.keys()
                if missing_fields:
                    missing = ", ".join(sorted(missing_fields))
                    raise ValueError(f"Missing {missing} in {dataset_path.name} at line {line_number}")

                test_id = test_case["id"]
                if not isinstance(test_id, str) or not test_id.strip():
                    raise ValueError(f"Invalid id in {dataset_path.name} at line {line_number}")
                if test_id in seen_ids:
                    raise ValueError(f"Duplicate synthetic test id: {test_id}")
                if test_case["category"] != expected_category:
                    raise ValueError(
                        f"Expected category {expected_category!r} in {dataset_path.name} at line {line_number}"
                    )
                if not isinstance(test_case["prompt"], str) or not test_case["prompt"].strip():
                    raise ValueError(f"Invalid prompt in {dataset_path.name} at line {line_number}")

                expected_keywords = test_case["expected_keywords"]
                if not isinstance(expected_keywords, list) or not expected_keywords or not all(
                    isinstance(keyword, str) and keyword.strip() for keyword in expected_keywords
                ):
                    raise ValueError(f"Invalid expected_keywords in {dataset_path.name} at line {line_number}")
                if not isinstance(test_case["reference_answer"], str) or not test_case["reference_answer"].strip():
                    raise ValueError(f"Invalid reference_answer in {dataset_path.name} at line {line_number}")

                seen_ids.add(test_id)
                test_cases.append(test_case)

    if not test_cases:
        raise ValueError("No synthetic test cases were loaded.")
    return test_cases


TEST_CASES = load_test_cases(SYNTHETIC_DATASETS)
MAX_EXAMPLES_PER_CATEGORY = 50  # set to None to score every loaded example
QUALITY_FLOOR = 0.80
QUALITY_TOLERANCE = 0.10
MAX_OPERATIONAL_REGRESSION = 0.20

if MAX_EXAMPLES_PER_CATEGORY:
    kept: dict[str, int] = {}
    capped_cases = []
    for case in TEST_CASES:
        seen = kept.get(case["category"], 0)
        if seen < MAX_EXAMPLES_PER_CATEGORY:
            kept[case["category"]] = seen + 1
            capped_cases.append(case)
    TEST_CASES = capped_cases

dataset_summary = " · ".join(
    f"{html.escape(category)}: {sum(case['category'] == category for case in TEST_CASES)}"
    for category in SYNTHETIC_DATASETS
)
display(HTML(
    f"<p><strong>{len(TEST_CASES)} shared synthetic examples</strong> &nbsp; {dataset_summary}</p>"
))


## 3. Run every test for each model

Every deployment runs every test `REPEATS` times. Live mode records the response, elapsed time, prompt tokens, completion tokens, reasoning tokens when available, total tokens, and estimated cost.

Live runs are sent through a thread pool of `MAX_CONCURRENT_REQUESTS` workers, which is the main lever on wall-clock time. Concurrency inflates measured latency through client-side and service-side queuing, so set it to `1` when the latency comparison has to be defensible, and keep it modest otherwise. Results are stored in submission order, so the comparison is identical either way.

In [3]:
def keyword_quality(response: str, expected_keywords: list[str]) -> float:
    text = response.casefold()
    return sum(keyword.casefold() in text for keyword in expected_keywords) / len(expected_keywords)


def call_foundry_model(model: dict, test_case: dict) -> dict:
    started = time.perf_counter()
    completion = openai_client.chat.completions.create(
        model=model["deployment"],
        messages=[{"role": "user", "content": test_case["prompt"]}],
        max_completion_tokens=MAX_COMPLETION_TOKENS,
    )
    latency_ms = (time.perf_counter() - started) * 1_000
    usage = completion.usage
    input_tokens = int(usage.prompt_tokens or 0)
    output_tokens = int(usage.completion_tokens or 0)
    usage_details = getattr(usage, "completion_tokens_details", None)
    reasoning_tokens = int(getattr(usage_details, "reasoning_tokens", 0) or 0)
    prices_available = model["input_price"] > 0 or model["output_price"] > 0
    estimated_cost = (
        (input_tokens * model["input_price"] + output_tokens * model["output_price"]) / 1_000_000
        if prices_available
        else None
    )
    return {
        "response": completion.choices[0].message.content or "",
        "latency_ms": latency_ms,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": reasoning_tokens,
        "total_tokens": int(usage.total_tokens or input_tokens + output_tokens),
        "estimated_cost_usd": estimated_cost,
        "status": "completed",
    }


def illustrative_observation(model_index: int, test_case: dict, repeat: int) -> dict:
    reference_response = test_case["reference_answer"]
    response = (
        reference_response
        if model_index == 0
        else test_case.get("illustrative_candidate_response", reference_response)
    )
    category_latency_ms = {
        "Summarisation": 620,
        "Classification": 430,
        "Extraction": 510,
    }
    case_offset = sum(ord(character) for character in test_case["id"]) % 5
    base_latency = category_latency_ms[test_case["category"]] + case_offset * 17
    base_input = max(16, round(len(test_case["prompt"].split()) * 1.3))
    base_output = max(4, round(len(response.split()) * 1.3))
    latency_factor = 1.0 if model_index == 0 else 0.64
    token_factor = 1.0 if model_index == 0 else 0.78
    input_tokens = round(base_input * token_factor) + repeat
    output_tokens = round(base_output * token_factor) + repeat
    return {
        "response": response,
        "latency_ms": base_latency * latency_factor + repeat * (19 if model_index == 0 else 11),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": 0,
        "total_tokens": input_tokens + output_tokens,
        "estimated_cost_usd": (0.00019 if model_index == 0 else 0.00008) + repeat * 0.000002,
        "status": "illustrative",
    }


jobs = [
    (model_index, model, test_case, repeat)
    for model_index, model in enumerate(MODELS)
    for test_case in TEST_CASES
    for repeat in range(1, REPEATS + 1)
]
total_runs = len(jobs)
smoke_result = None
if RUN_LIVE_EVALUATION:
    smoke_model = MODELS[0]
    smoke_test = TEST_CASES[0]
    print(
        f"[1/{total_runs}] Smoke test: {smoke_model['role']} / {smoke_test['id']} "
        f"(timeout {REQUEST_TIMEOUT_SECONDS:.0f}s)",
        flush=True,
    )
    try:
        smoke_result = call_foundry_model(smoke_model, smoke_test)
    except Exception as exc:
        raise RuntimeError(
            "The live smoke test failed, so the full comparison was not started. "
            "Check the deployment names, credentials, endpoint, and network access."
        ) from exc
    print(f"[1/{total_runs}] Smoke test completed in {smoke_result['latency_ms'] / 1_000:.1f}s", flush=True)


def run_job(job: tuple) -> dict:
    model_index, model, test_case, repeat = job
    try:
        if RUN_LIVE_EVALUATION:
            result = call_foundry_model(model, test_case)
        else:
            result = illustrative_observation(model_index, test_case, repeat)
    except Exception as exc:
        result = {
            "response": "",
            "latency_ms": None,
            "input_tokens": 0,
            "output_tokens": 0,
            "reasoning_tokens": 0,
            "total_tokens": 0,
            "estimated_cost_usd": None,
            "status": f"failed: {type(exc).__name__}: {exc}",
        }
    return {
        "role": model["role"],
        "deployment": model["deployment"],
        "test_id": test_case["id"],
        "category": test_case["category"],
        "repeat": repeat,
        "quality": keyword_quality(result["response"], test_case["expected_keywords"]),
        **result,
    }


started_at = time.perf_counter()
observations = [None] * total_runs
concurrency = MAX_CONCURRENT_REQUESTS if RUN_LIVE_EVALUATION else 1

if concurrency > 1:
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        pending = {pool.submit(run_job, job): index for index, job in enumerate(jobs)}
        for completed, future in enumerate(as_completed(pending), start=1):
            observations[pending[future]] = future.result()
            if completed % PROGRESS_EVERY == 0 or completed == total_runs:
                elapsed = time.perf_counter() - started_at
                print(f"[{completed}/{total_runs}] {elapsed:.0f}s elapsed", flush=True)
else:
    for index, job in enumerate(jobs):
        observations[index] = run_job(job)
        if RUN_LIVE_EVALUATION and ((index + 1) % PROGRESS_EVERY == 0 or index + 1 == total_runs):
            elapsed = time.perf_counter() - started_at
            print(f"[{index + 1}/{total_runs}] {elapsed:.0f}s elapsed", flush=True)

responses_df = pd.DataFrame(observations)
failures = responses_df[responses_df["status"].str.startswith("failed")]
completed_count = len(responses_df) - len(failures)
status_color = "#087f5b" if failures.empty else "#b42318"
display(HTML(
    f"<div style='padding:10px 14px;border-left:5px solid {status_color};background:#f7f7f5'>"
    f"<strong>{completed_count} of {len(responses_df)} runs collected</strong>"
    f" in {time.perf_counter() - started_at:.0f}s"
    + (f" with {concurrency} concurrent requests" if concurrency > 1 else "")
    + ("" if failures.empty else f"<br>{len(failures)} runs failed; review the status in the exported responses.")
    + "</div>"
))

[1/630] Smoke test: baseline / summarisation-0001 (timeout 60s)


[1/630] Smoke test completed in 13.7s
[25/630] 7s elapsed
[50/630] 14s elapsed
[75/630] 21s elapsed
[100/630] 29s elapsed
[125/630] 40s elapsed
[150/630] 58s elapsed
[175/630] 71s elapsed
[200/630] 80s elapsed
[225/630] 87s elapsed
[250/630] 93s elapsed
[275/630] 99s elapsed
[300/630] 105s elapsed
[325/630] 108s elapsed
[350/630] 111s elapsed
[375/630] 114s elapsed
[400/630] 116s elapsed
[425/630] 120s elapsed
[450/630] 124s elapsed
[475/630] 127s elapsed
[500/630] 129s elapsed
[525/630] 131s elapsed
[550/630] 133s elapsed
[575/630] 137s elapsed
[600/630] 139s elapsed
[625/630] 142s elapsed
[630/630] 143s elapsed


## 4. Build the comparative decision

Every repeated run of every example is aggregated into one row per workload category, so each model is represented by its mean quality, latency, token use, and estimated cost across that category.

Direction matters: higher mean quality is better, while lower mean latency, lower mean tokens, and lower mean cost are better. An efficiency measure counts as a win only when the candidate is strictly lower than the baseline; a measure without prices on both models is skipped.

The quality gate adapts to the incumbent. When the baseline clears the 80% floor, the candidate must clear it too and stay within the 10 pp tolerance of the baseline. When the baseline itself misses the floor, the workload is one this test set scores harshly for both models, so the absolute floor is dropped and the candidate only has to hold parity within 5 pp of the baseline; every verdict from that path is labelled to show the floor was unmet.

There is no standalone composite score. A category is marked **Move** when the candidate passes the quality gate and wins at least two available efficiency measures. **Pilot** means the gate is passed with exactly one efficiency win. Any efficiency measure more than 20% worse than the baseline, or a failed quality gate, forces **Hold**.


In [4]:
successful_df = responses_df[~responses_df["status"].str.startswith("failed")].copy()

# Only examples that both models answered at least once can be compared fairly.
comparable_ids = set(successful_df.loc[successful_df["role"] == "baseline", "test_id"]) & set(
    successful_df.loc[successful_df["role"] == "candidate", "test_id"]
)
excluded_ids = sorted(set(responses_df["test_id"]) - comparable_ids)
successful_df = successful_df[successful_df["test_id"].isin(comparable_ids)]
if successful_df.empty:
    raise RuntimeError("No example produced a successful run for both models.")


def coefficient_of_variation(values: pd.Series) -> float:
    mean = values.mean()
    return float(values.std(ddof=0) / mean) if mean else 0.0


# Every repeat of every example collapses into one row per model per category.
model_summary = (
    successful_df.groupby(["role", "deployment", "category"], as_index=False)
    .agg(
        examples=("test_id", "nunique"),
        runs=("test_id", "size"),
        quality=("quality", "mean"),
        latency_ms=("latency_ms", "mean"),
        total_tokens=("total_tokens", "mean"),
        cost_usd=("estimated_cost_usd", "mean"),
        latency_cv=("latency_ms", coefficient_of_variation),
    )
)

baseline = (
    model_summary[model_summary["role"] == "baseline"]
    .drop(columns=["role", "deployment"])
    .rename(columns={
        "examples": "baseline_examples",
        "runs": "baseline_runs",
        "quality": "baseline_quality",
        "latency_ms": "baseline_latency_ms",
        "total_tokens": "baseline_total_tokens",
        "cost_usd": "baseline_cost_usd",
        "latency_cv": "baseline_latency_cv",
    })
)
candidate = (
    model_summary[model_summary["role"] == "candidate"]
    .drop(columns=["role", "deployment"])
    .rename(columns={
        "examples": "candidate_examples",
        "runs": "candidate_runs",
        "quality": "candidate_quality",
        "latency_ms": "candidate_latency_ms",
        "total_tokens": "candidate_total_tokens",
        "cost_usd": "candidate_cost_usd",
        "latency_cv": "candidate_latency_cv",
    })
)
comparison_df = baseline.merge(candidate, on="category", validate="one_to_one")

# Without configured prices the cost columns arrive as object dtype full of None.
for column in ("baseline_cost_usd", "candidate_cost_usd"):
    comparison_df[column] = pd.to_numeric(comparison_df[column], errors="coerce")

if comparison_df.empty:
    raise RuntimeError("No workload category has comparable results for both models.")
if excluded_ids:
    display(HTML(
        "<div style='padding:10px 14px;border-left:5px solid #b26a00;background:#fdf7ee'>"
        f"<strong>{len(excluded_ids)} of {responses_df['test_id'].nunique()} examples excluded</strong>"
        " because at least one model returned no successful run for them."
        "</div>"
    ))

comparison_df["quality_delta_pp"] = (comparison_df["candidate_quality"] - comparison_df["baseline_quality"]) * 100
comparison_df["latency_delta_pct"] = (comparison_df["candidate_latency_ms"] / comparison_df["baseline_latency_ms"] - 1) * 100
comparison_df["token_delta_pct"] = (comparison_df["candidate_total_tokens"] / comparison_df["baseline_total_tokens"] - 1) * 100
comparison_df["cost_delta_pct"] = float("nan")
cost_comparable = comparison_df["baseline_cost_usd"].notna() & comparison_df["candidate_cost_usd"].notna()
comparison_df.loc[cost_comparable, "cost_delta_pct"] = (
    comparison_df.loc[cost_comparable, "candidate_cost_usd"]
    / comparison_df.loc[cost_comparable, "baseline_cost_usd"]
    - 1
) * 100

# Quality is higher-is-better; every efficiency measure below is lower-is-better.
EFFICIENCY_MEASURES = {"latency_ms": "latency", "total_tokens": "tokens", "cost_usd": "cost"}
# Applied instead of the absolute floor when the incumbent itself cannot clear it.
PARITY_TOLERANCE = 0.05


def displacement_decision(row: pd.Series) -> pd.Series:
    baseline_meets_floor = row["baseline_quality"] >= QUALITY_FLOOR
    if baseline_meets_floor:
        quality_failure = None
        if row["candidate_quality"] < QUALITY_FLOOR:
            quality_failure = (
                f"Mean quality {row['candidate_quality']:.0%} is below the {QUALITY_FLOOR:.0%} absolute floor"
            )
        elif row["candidate_quality"] < row["baseline_quality"] - QUALITY_TOLERANCE:
            quality_failure = (
                f"Mean quality {row['candidate_quality']:.0%} trails the baseline {row['baseline_quality']:.0%} "
                f"by more than the {QUALITY_TOLERANCE * 100:g} pp tolerance"
            )
        caveat = ""
    else:
        quality_failure = None
        if row["candidate_quality"] < row["baseline_quality"] - PARITY_TOLERANCE:
            quality_failure = (
                f"Neither model met the {QUALITY_FLOOR:.0%} floor and mean quality {row['candidate_quality']:.0%} "
                f"trails the baseline {row['baseline_quality']:.0%} by more than the {PARITY_TOLERANCE * 100:g} pp "
                "parity tolerance"
            )
        caveat = f"; neither model met the {QUALITY_FLOOR:.0%} quality floor"

    wins, ties, regressions = [], [], []
    for measure, label in EFFICIENCY_MEASURES.items():
        baseline_value = row[f"baseline_{measure}"]
        candidate_value = row[f"candidate_{measure}"]
        if pd.isna(baseline_value) or pd.isna(candidate_value) or baseline_value <= 0:
            continue
        if candidate_value < baseline_value:
            wins.append(label)
        elif candidate_value == baseline_value:
            ties.append(label)
        if candidate_value > baseline_value * (1 + MAX_OPERATIONAL_REGRESSION):
            regressions.append(label)

    if quality_failure:
        return pd.Series({"verdict": "HOLD", "reason": quality_failure})
    if regressions:
        return pd.Series({
            "verdict": "HOLD",
            "reason": f"Candidate exceeded the {MAX_OPERATIONAL_REGRESSION:.0%} regression allowance on mean {', '.join(regressions)}",
        })
    if len(wins) >= 2:
        return pd.Series({"verdict": "MOVE", "reason": f"Quality held at parity and lower mean {', '.join(wins)}{caveat}"})
    if len(wins) == 1:
        return pd.Series({"verdict": "PILOT", "reason": f"Quality held at parity with lower mean {wins[0]} only{caveat}"})
    if ties:
        return pd.Series({"verdict": "HOLD", "reason": f"Quality held at parity but {', '.join(ties)} matched the baseline"})
    return pd.Series({"verdict": "HOLD", "reason": "Quality held at parity but no efficiency measure improved"})


comparison_df[["verdict", "reason"]] = comparison_df.apply(displacement_decision, axis=1)
comparison_df


,category,baseline_examples,baseline_runs,baseline_quality,baseline_latency_ms,baseline_total_tokens,baseline_cost_usd,baseline_latency_cv,candidate_examples,candidate_runs,...,candidate_latency_ms,candidate_total_tokens,candidate_cost_usd,candidate_latency_cv,quality_delta_pp,latency_delta_pct,token_delta_pct,cost_delta_pct,verdict,reason
0,Classification,50,150,0.960000,1461.006386,48.900000,0.000016,0.263697,50,150,...,773.231329,44.140000,0.000009,0.929537,-3.333333,-47.075431,-9.734151,-40.934591,MOVE,"Quality held at parity and lower mean latency,..."
1,Extraction,5,15,0.977778,1427.758652,66.466667,0.000038,0.219779,5,15,...,1031.907291,70.866667,0.000023,0.708940,-1.111111,-27.725369,6.619860,-39.502096,MOVE,"Quality held at parity and lower mean latency,..."
2,Summarisation,50,145,0.728736,2156.681043,79.337931,0.000048,1.746765,50,150,...,1023.191112,76.746667,0.000025,0.844643,-2.762452,-52.557143,-3.266110,-47.807703,MOVE,"Quality held at parity and lower mean latency,..."


## 5. Visual displacement scorecard

One panel per workload category, each based on the mean of every example and repeat in that category. Longer bars indicate better performance: quality uses its absolute percentage, while latency, token, and cost bars are normalized so the more efficient model has the longer bar. The decision badge is the displacement result, not a separate arbitrary score.


In [5]:
BASELINE_COLOR = MODELS[0]["color"]
CANDIDATE_COLOR = MODELS[1]["color"]
VERDICT_COLORS = {"MOVE": "#087f5b", "PILOT": "#b26a00", "HOLD": "#b42318"}
PORTFOLIO_BAR_COLOR = "#003a8c"


def clamp_percent(value: float) -> float:
    return max(2.0, min(100.0, float(value)))


def efficiency_widths(baseline_value: float, candidate_value: float) -> tuple[float, float]:
    best = min(baseline_value, candidate_value)
    return clamp_percent(best / baseline_value * 100), clamp_percent(best / candidate_value * 100)


def paired_bars(
    title: str,
    baseline_width: float,
    candidate_width: float,
    baseline_label: str,
    candidate_label: str,
) -> str:
    baseline_name = html.escape(MODELS[0]["deployment"])
    candidate_name = html.escape(MODELS[1]["deployment"])
    return f"""
    <div class="metric">
      <div class="metric-title">{html.escape(title)}</div>
      <div class="bar-row"><span title="{baseline_name}">Baseline</span><div class="track"><div class="bar baseline" style="width:{clamp_percent(baseline_width):.1f}%"></div></div><strong>{html.escape(baseline_label)}</strong></div>
      <div class="bar-row"><span title="{candidate_name}">Candidate</span><div class="track"><div class="bar candidate" style="width:{clamp_percent(candidate_width):.1f}%"></div></div><strong>{html.escape(candidate_label)}</strong></div>
    </div>
    """


def workload_panel(row: pd.Series) -> str:
    latency_widths = efficiency_widths(row["baseline_latency_ms"], row["candidate_latency_ms"])
    token_widths = efficiency_widths(row["baseline_total_tokens"], row["candidate_total_tokens"])
    cost_is_available = pd.notna(row["baseline_cost_usd"]) and pd.notna(row["candidate_cost_usd"])

    metrics = [
        paired_bars(
            "Mean quality",
            row["baseline_quality"] * 100,
            row["candidate_quality"] * 100,
            f"{row['baseline_quality']:.0%}",
            f"{row['candidate_quality']:.0%}",
        ),
        paired_bars(
            "Mean latency efficiency",
            *latency_widths,
            f"{row['baseline_latency_ms']:,.0f} ms",
            f"{row['candidate_latency_ms']:,.0f} ms",
        ),
        paired_bars(
            "Mean token efficiency",
            *token_widths,
            f"{row['baseline_total_tokens']:,.0f}",
            f"{row['candidate_total_tokens']:,.0f}",
        ),
    ]
    if cost_is_available:
        cost_widths = efficiency_widths(row["baseline_cost_usd"], row["candidate_cost_usd"])
        metrics.append(paired_bars(
            "Mean cost efficiency",
            *cost_widths,
            f"${row['baseline_cost_usd']:.6f}",
            f"${row['candidate_cost_usd']:.6f}",
        ))
    else:
        metrics.append("<div class='metric unavailable'><div class='metric-title'>Mean cost efficiency</div>Configure per-model prices to compare cost.</div>")

    color = VERDICT_COLORS[row["verdict"]]
    sample = f"{int(row['baseline_examples']):,} examples · {int(row['baseline_runs']):,} runs per model"
    return f"""
    <section class="workload">
      <div class="workload-head">
        <div><div class="eyebrow">WORKLOAD</div><h3>{html.escape(row['category'])}</h3><div class="sample">{html.escape(sample)}</div></div>
        <span class="verdict" style="background:{color}">{row['verdict']}</span>
      </div>
      <div class="metric-grid">{''.join(metrics)}</div>
      <div class="reason">{html.escape(row['reason'])}</div>
    </section>
    """


verdict_counts = comparison_df["verdict"].value_counts()
move_count = int(verdict_counts.get("MOVE", 0))
pilot_count = int(verdict_counts.get("PILOT", 0))
hold_count = int(verdict_counts.get("HOLD", 0))
if hold_count == 0 and move_count + pilot_count == len(comparison_df):
    portfolio_verdict = "CANDIDATE CAN DISPLACE BASELINE"
elif move_count + pilot_count > 0:
    portfolio_verdict = "SELECTIVE DISPLACEMENT"
else:
    portfolio_verdict = "KEEP BASELINE"

scorecard_html = f"""
<style>
.scorecard {{font-family:Segoe UI,Arial,sans-serif;color:#1f2933;background:#f7f8f5;padding:22px;max-width:1120px}}
.summary {{border-top:6px solid {PORTFOLIO_BAR_COLOR};padding:18px 0 20px;display:flex;justify-content:space-between;gap:24px;align-items:end}}
.summary h2 {{font-size:26px;margin:2px 0 6px;letter-spacing:0}}
.summary p {{margin:0;color:#59636e}}
.counts {{display:flex;gap:10px;flex-wrap:wrap}}
.count {{min-width:72px;padding:8px 12px;background:#fff;border:1px solid #d9ddd7;text-align:center}}
.count strong {{display:block;font-size:21px}}
.legend {{display:flex;gap:18px;padding:10px 0 18px;border-bottom:1px solid #d9ddd7;font-size:13px}}
.dot {{display:inline-block;width:10px;height:10px;margin-right:6px}}
.workload {{padding:22px 0;border-bottom:1px solid #d9ddd7}}
.workload-head {{display:flex;justify-content:space-between;align-items:center;margin-bottom:15px}}
.workload h3 {{margin:0;font-size:20px;letter-spacing:0}}
.eyebrow {{font-size:10px;font-weight:700;color:#6b7280}}
.sample {{font-size:12px;color:#6b7280;margin-top:3px}}
.verdict {{color:#fff;font-weight:800;padding:7px 12px;font-size:12px}}
.metric-grid {{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:14px 24px}}
.metric {{background:#fff;border:1px solid #e0e3de;padding:12px}}
.metric-title {{font-size:12px;font-weight:800;margin-bottom:8px}}
.bar-row {{display:grid;grid-template-columns:72px minmax(90px,1fr) 78px;align-items:center;gap:8px;font-size:11px;margin:6px 0}}
.bar-row strong {{text-align:right;font-size:11px}}
.track {{height:12px;background:#e9ece8;overflow:hidden}}
.bar {{height:100%}}
.baseline {{background:{BASELINE_COLOR}}}
.candidate {{background:{CANDIDATE_COLOR}}}
.reason {{margin-top:12px;font-size:13px;color:#4b5563}}
.unavailable {{color:#6b7280;font-size:12px}}
@media (max-width:760px) {{.summary {{display:block}} .counts {{margin-top:14px}} .metric-grid {{grid-template-columns:1fr}}}}
</style>
<div class="scorecard">
  <div class="summary">
    <div><div class="eyebrow">PORTFOLIO DECISION</div><h2>{portfolio_verdict}</h2><p>{html.escape(MODELS[1]['deployment'])} compared with {html.escape(MODELS[0]['deployment'])}</p></div>
    <div class="counts"><div class="count"><strong>{move_count}</strong>Move</div><div class="count"><strong>{pilot_count}</strong>Pilot</div><div class="count"><strong>{hold_count}</strong>Hold</div></div>
  </div>
  <div class="legend"><span><i class="dot" style="background:{BASELINE_COLOR}"></i>Baseline: {html.escape(MODELS[0]['deployment'])}</span><span><i class="dot" style="background:{CANDIDATE_COLOR}"></i>Candidate: {html.escape(MODELS[1]['deployment'])}</span></div>
  {''.join(workload_panel(row) for _, row in comparison_df.iterrows())}
</div>
"""
display(HTML(scorecard_html))

## 6. Export the comparison

The displayed scorecard is saved as a standalone HTML report, the decision summary as CSV, and the repeated model responses as JSON.

In [6]:
RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")
html_path = KIT_ROOT / "reports" / f"two-model-displacement-scorecard-{RUN_STAMP}.html"
summary_path = KIT_ROOT / "reports" / f"two-model-displacement-summary-{RUN_STAMP}.csv"
responses_path = KIT_ROOT / "results" / f"two-model-evaluation-responses-{RUN_STAMP}.json"
html_path.parent.mkdir(parents=True, exist_ok=True)
responses_path.parent.mkdir(parents=True, exist_ok=True)

html_path.write_text(scorecard_html, encoding="utf-8")
comparison_df.to_csv(summary_path, index=False)
responses_path.write_text(json.dumps(observations, indent=2), encoding="utf-8")

for resource in (openai_client, project_client, credential):
    close = getattr(resource, "close", None)
    if callable(close):
        close()

display(HTML(
    "<p><strong>Comparison exported</strong></p>"
    f"<code>{html.escape(str(html_path))}</code><br>"
    f"<code>{html.escape(str(summary_path))}</code><br>"
    f"<code>{html.escape(str(responses_path))}</code>"
))